In [6]:
def get_matches_today_data():
    import statsapi
    import mlbstatsapi
    from datetime import datetime

    # Get today's schedule
    matches_today = []

    # get the proper formatted date
    mlb_date = datetime.now().strftime("%m/%d/%Y")

    # get the schedule as a dictionary for today
    schedule = statsapi.schedule(start_date=mlb_date, end_date=mlb_date)

    # iterate through each game of the schedule
    for x in schedule:

        # initialize game data dictionary
        game_data = {}
        
        # away_name
        game_data.update({'away_name': x.get('away_name')}) 
        
        # home_name
        game_data.update({'home_name': x.get('home_name')})
        
        # away_id
        game_data.update({'away_id': x.get('away_id')})

        away_team_leaders_hr = []
        # add top away team guys here
        away_leaders = statsapi.team_leader_data(x.get('away_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in away_leaders:
            away_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'away_team_leaders_hr': away_team_leaders_hr})

        home_team_leaders_hr = []
        # add top away team guys here
        home_leaders = statsapi.team_leader_data(x.get('home_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in home_leaders:
            home_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'home_team_leaders_hr': home_team_leaders_hr})

        # home_id
        game_data.update({'home_id': x.get('home_id')})
        # home_probable_pitcher
        game_data.update({'home_probable_pitcher': x.get('home_probable_pitcher')})
        # away_probable_pitcher
        game_data.update({'away_probable_pitcher': x.get('away_probable_pitcher')})

        matches_today.append(game_data)


    mlb = mlbstatsapi.Mlb()

    for x in matches_today:
  
        away_probable_pitcher = x.get('away_probable_pitcher')
        
        # Check if away_probable_pitcher is valid
        if not away_probable_pitcher:
            print(f"Warning: Missing away_probable_pitcher for game: {x}")
            continue  # Skip this game if no pitcher is available

        pitcher_ids = mlb.get_people_id(away_probable_pitcher)
        
        # Check if pitcher_ids is not empty
        if not pitcher_ids:
            print(f"Warning: No pitcher ID found for {away_probable_pitcher}")
            continue  # Skip this game if no pitcher ID is found

        pitcher_id = pitcher_ids[0]  # Safely access the first element

        BvP = []
        for y in x.get('home_team_leaders_hr', []):  # Default to an empty list if key is missing
            batter_id = mlb.get_people_id(y.get('name'))[0]

            stats = ['vsPlayer']
            group = ['hitting']
            params = {'opposingPlayerId': pitcher_id, 'season': 2025}

            try:
                stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
                vs_player_total = stats['hitting']['vsplayertotal']
                for split in vs_player_total.splits:
                    p_id = mlb.get_person(pitcher_id)
                    b_id = mlb.get_person(batter_id)
                    
                    bvp_matchup = f"pitcher: {p_id.__dict__.get('fullname')} vs batter: {b_id.__dict__.get('fullname')}"
                    dict2 = {'bvp_stats': split.stat.__dict__}
                    dict2.update({'bvp_matchup': bvp_matchup})
                    dict2.update({'pitcher': p_id.__dict__.get('fullname')})
                    dict2.update({'batter': b_id.__dict__.get('fullname')})
                    BvP.append(dict2)

            except KeyError as e:
                print(f"KeyError: {e}. Skipping this player. Stats: {stats}")
            except Exception as e:
                print(f"Unexpected error: {e}. Skipping this player.")
        


        
        home_probable_pitcher = x.get('home_probable_pitcher')
        
        # Check if home_probable_pitcher is valid
        if not home_probable_pitcher:
            print(f"Warning: Missing home_probable_pitcher for game: {x}")
            continue  # Skip this game if no pitcher is available

        pitcher_ids = mlb.get_people_id(home_probable_pitcher)
        
        # Check if pitcher_ids is not empty
        if not pitcher_ids:
            print(f"Warning: No pitcher ID found for {home_probable_pitcher}")
            continue  # Skip this game if no pitcher ID is found

        pitcher_id = pitcher_ids[0]  # Safely access the first element

        for y in x.get('away_team_leaders_hr', []):  # Default to an empty list if key is missing
            batter_id = mlb.get_people_id(y.get('name'))[0]

            stats = ['vsPlayer']
            group = ['hitting']
            params = {'opposingPlayerId': pitcher_id, 'season': 2025}

            try:
                stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
                vs_player_total = stats['hitting']['vsplayertotal']
                for split in vs_player_total.splits:
                    p_id = mlb.get_person(pitcher_id)
                    b_id = mlb.get_person(batter_id)
                    
                    bvp_matchup = f"pitcher: {p_id.__dict__.get('fullname')} vs batter: {b_id.__dict__.get('fullname')}"
                    dict2 = {'bvp_stats': split.stat.__dict__}
                    dict2.update({'bvp_matchup': bvp_matchup})
                    dict2.update({'pitcher': p_id.__dict__.get('fullname')})
                    dict2.update({'batter': b_id.__dict__.get('fullname')})
                    BvP.append(dict2)

            except KeyError as e:
                print(f"KeyError: {e}. Skipping this player. Stats: {stats}")
            except Exception as e:
                print(f"Unexpected error: {e}. Skipping this player.")
        
        # Add the BvP stats to the matches_today dictionary
        x.update({'BvP_stats': BvP})

    return matches_today

# todays_matches = get_matches_today_data()

# import sys
# import os


# sys.stdout = open(os.devnull, 'w')

# Call your function
todays_matches = get_matches_today_data()

# # Restore output
# sys.stdout = sys.__stdout__

# # Print a readable output of each item in the dictionary of the list matches_today
# for match in todays_matches:
#     print(f"Match: AWAY: {match['away_name']} vs HOME: {match['home_name']}")
#     print(f"Away Team ID: {match['away_id']}, Home Team ID: {match['home_id']}")
#     print(f"Away Probable Pitcher: {match['away_probable_pitcher']}")
#     print(f"Home Probable Pitcher: {match['home_probable_pitcher']}")
    
#     print("Away Team Leaders in Home Runs:")
#     for leader in match['away_team_leaders_hr']:
#         print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
#     print("Home Team Leaders in Home Runs:")
#     for leader in match['home_team_leaders_hr']:
#         print(f"  {leader['name']}: {leader['homeRuns']} HR")
    
#     if 'BvP_stats' in match:
#         print("Batter vs Pitcher Stats:")
#         for bvp in match['BvP_stats']:
#             print(f"  Matchup: {bvp['bvp_matchup']}")
#             for stat, value in bvp['bvp_stats'].items():
#                 print(f"    {stat}: {value}")
    
#     print("\n")  # Print a newline for better readability between matches



https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/682829/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/682829
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/669720/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/669720
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/680574/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/680574
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/682622/stats
https://statsapi.mlb.com/api/v1/people/571945
https://statsapi.mlb.com/api/v1/people/682622
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/642851/stats
KeyError: 'hitting'. Skipping this player. Stats: {}
https://statsapi.mlb.com/

In [8]:

for match in todays_matches:

    if 'BvP_stats' in match:
        print("Batter vs Pitcher Stats:")
        print()
        print(f"Away Probable Pitcher: {match['away_probable_pitcher']}")
        print(f"Home Probable Pitcher: {match['home_probable_pitcher']}")
        print()
        for bvp in match['BvP_stats']:
            batter_name = bvp['bvp_matchup']
            print(f"Match: AWAY: {match['away_name']} vs HOME: {match['home_name']}")
            print(f"Batter:  {bvp['batter']:>10}")
            print(f"Pitcher: {bvp['pitcher']:>10}")
            print(f"obp: {bvp['bvp_stats'].get('obp', 'N/A'):>10}")
            print(f"ops: {bvp['bvp_stats'].get('ops', 'N/A'):>10}")
            print(f"avg: {bvp['bvp_stats'].get('avg', 'N/A'):>10}")
            print(f" AB: {bvp['bvp_stats'].get('atbats', 'N/A'):>10}")
            print(f"  H: {bvp['bvp_stats'].get('hits', 'N/A'):>10}")
            print(f" HR: {bvp['bvp_stats'].get('homeruns', 'N/A'):>10}")
            print(f"RBI: {bvp['bvp_stats'].get('rbi', 'N/A'):>10}")
            print(f"AVG: {bvp['bvp_stats'].get('avg', 'N/A'):>10}")
            print()

    print("\n")  # Print a newline for better readability between matches


Batter vs Pitcher Stats:

Away Probable Pitcher: Miles Mikolas
Home Probable Pitcher: Brady Singer

Match: AWAY: St. Louis Cardinals vs HOME: Cincinnati Reds
Batter:  Elly De La Cruz
Pitcher: Miles Mikolas
obp:       .417
ops:      1.053
avg:       .364
 AB:         11
  H:          4
 HR:          0
RBI:          3
AVG:       .364

Match: AWAY: St. Louis Cardinals vs HOME: Cincinnati Reds
Batter:  Austin Hays
Pitcher: Miles Mikolas
obp:       .000
ops:       .000
avg:       .000
 AB:          3
  H:          0
 HR:          0
RBI:          0
AVG:       .000

Match: AWAY: St. Louis Cardinals vs HOME: Cincinnati Reds
Batter:  Matt McLain
Pitcher: Miles Mikolas
obp:       .333
ops:      1.000
avg:       .333
 AB:          6
  H:          2
 HR:          0
RBI:          0
AVG:       .333

Match: AWAY: St. Louis Cardinals vs HOME: Cincinnati Reds
Batter:  Noelvi Marte
Pitcher: Miles Mikolas
obp:       .333
ops:       .833
avg:       .333
 AB:          6
  H:          2
 HR:          0
RBI: